In [ ]:
"""Load the HAM10000 dataset with PyTorch and create stratified data splits.

Expected layout in the project root:

data/
  HAM10000_metadata.csv
  HAM10000_images_part_1/
    ISIC_....jpg
  HAM10000_images_part_2/
    ISIC_....jpg
"""
# Neccessary imports
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    recall_score,
    )
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms

# Model 1 imports
from model1 import (
    build_resnet50_feature_extractor,
    build_svm_estimator,
    extract_cnn_features,
    train_svm,
)

# Model 2 imports
from densenet import predict_densenet, run_densenet_pipeline

In [ ]:
def build_default_transforms(image_size: int = 224):
    """Create default train/test transform pipelines."""
    train_transform = transforms.Compose(
        [
            transforms.Resize((image_size, image_size)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(),
            transforms.ToTensor(),
        ]
    )

    test_transform = transforms.Compose(
        [
            transforms.Resize((image_size, image_size)),
            transforms.CenterCrop((image_size, image_size)),
            transforms.ToTensor(),
        ]
    )

    return train_transform, test_transform


class HAM10000Dataset(Dataset):
    """PyTorch dataset for HAM10000 images and labels."""

    def __init__(self, data_dir: str | Path = "data", transform=None):
        self.data_dir = Path(data_dir)
        self.transform = transform
        self.metadata = self._load_metadata(self.data_dir)
        self.class_names = sorted(self.metadata["dx"].unique())
        self.class_to_idx = {label: index for index, label in enumerate(self.class_names)}
        self.metadata["label"] = self.metadata["dx"].map(self.class_to_idx)

    @staticmethod
    def _load_metadata(data_dir: Path) -> pd.DataFrame:
        from zipfile import BadZipFile, ZipFile, is_zipfile

        metadata_csv = data_dir / "HAM10000_metadata.csv"
        part_1_zip = data_dir / "HAM10000_images_part_1.zip"
        part_2_zip = data_dir / "HAM10000_images_part_2.zip"

        if not metadata_csv.exists():
            raise FileNotFoundError(f"Missing required file: {metadata_csv.resolve()}")

        metadata_df = pd.read_csv(metadata_csv)
        required_columns = {"image_id", "dx"}
        missing_columns = required_columns.difference(metadata_df.columns)
        if missing_columns:
            raise ValueError(
                "Missing required columns in HAM10000_metadata.csv: "
                + ", ".join(sorted(missing_columns))
            )

        def ensure_extracted_dir(zip_path: Path, extracted_dir: Path) -> Path:
            existing_images = list(extracted_dir.rglob("*.jpg")) if extracted_dir.exists() else []
            if existing_images:
                return extracted_dir

            if not zip_path.exists():
                raise FileNotFoundError(
                    f"Neither extracted images nor zip archive found for {extracted_dir.name}. "
                    f"Expected zip at: {zip_path.resolve()}"
                )

            if not is_zipfile(zip_path):
                raise BadZipFile(
                    f"Invalid zip archive at {zip_path.resolve()} (cwd={Path.cwd()})."
                )

            extracted_dir.mkdir(parents=True, exist_ok=True)
            try:
                with ZipFile(zip_path, "r") as zip_ref:
                    zip_ref.extractall(extracted_dir)
            except BadZipFile as exc:
                raise BadZipFile(
                    f"Could not open archive {zip_path.resolve()} (cwd={Path.cwd()})."
                ) from exc

            return extracted_dir

        extracted_part_1 = ensure_extracted_dir(part_1_zip, data_dir / "HAM10000_images_part_1")
        extracted_part_2 = ensure_extracted_dir(part_2_zip, data_dir / "HAM10000_images_part_2")

        image_files = list(extracted_part_1.rglob("*.jpg")) + list(extracted_part_2.rglob("*.jpg"))
        image_map = {image.stem: str(image.resolve()) for image in image_files}
        metadata_df["image_path"] = metadata_df["image_id"].map(image_map)

        missing_paths = metadata_df["image_path"].isna()
        if missing_paths.any():
            missing_count = int(missing_paths.sum())
            raise FileNotFoundError(
                f"Found {missing_count} metadata rows without a matching image file."
            )

        return metadata_df.reset_index(drop=True)

    def __len__(self) -> int:
        return len(self.metadata)

    def __getitem__(self, index: int):
        row = self.metadata.iloc[index]
        image = Image.open(row["image_path"]).convert("RGB")
        label = int(row["label"])

        if self.transform is not None:
            image = self.transform(image)

        return image, label


def create_data_splits(
    dataset: HAM10000Dataset,
    test_size: float = 0.2,
    random_state: int = 42,
):
    """Create stratified train/test subsets with preserved class proportions."""
    indices = dataset.metadata.index.to_numpy()
    labels = dataset.metadata["label"].to_numpy()

    train_indices, test_indices = train_test_split(
        indices,
        test_size=test_size,
        random_state=random_state,
        stratify=labels,
    )
    train_dataset = Subset(dataset, train_indices.tolist())
    test_dataset = Subset(dataset, test_indices.tolist())

    return train_dataset, test_dataset


def create_dataloaders(
    data_dir: str | Path = "data",
    batch_size: int = 64,
    test_size: float = 0.2,
    image_size: int = 224,
    random_state: int = 42,
    train_transform=None,
    test_transform=None,
):
    """Build dataset, stratified splits, and batch dataloaders with separate transforms."""
    if train_transform is None or test_transform is None:
        default_train_transform, default_test_transform = build_default_transforms(image_size=image_size)
        train_transform = train_transform or default_train_transform
        test_transform = test_transform or default_test_transform

    base_dataset = HAM10000Dataset(data_dir=data_dir, transform=None)
    train_subset, test_subset = create_data_splits(
        base_dataset,
        test_size=test_size,
        random_state=random_state,
    )

    train_dataset_with_transform = HAM10000Dataset(data_dir=data_dir, transform=train_transform)
    test_dataset_with_transform = HAM10000Dataset(data_dir=data_dir, transform=test_transform)

    train_dataset = Subset(train_dataset_with_transform, train_subset.indices)
    test_dataset = Subset(test_dataset_with_transform, test_subset.indices)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return base_dataset, train_dataset, test_dataset, train_loader, test_loader

In [ ]:
if __name__ == "__main__":

    print("Loading HAM10000 dataset and creating dataloaders...")

    train_transform, test_transform = build_default_transforms(image_size=224)

    dataset, train_dataset, test_dataset, train_loader, test_loader = create_dataloaders(
        data_dir="data",
        batch_size=64,
        test_size=0.2,
        image_size=224,
        random_state=42,
        train_transform=train_transform,
        test_transform=test_transform,
    )

    print(f"Total samples: {len(dataset)}")
    print(f"Training samples: {len(train_dataset)}")
    print(f"Testing samples: {len(test_dataset)}")
    print(f"Classes: {dataset.class_names}")
    print(f"Classes in training set: {train_dataset.dataset.class_names}")
    print(f"Classes in testing set: {test_dataset.dataset.class_names}")

    batch_images, batch_labels = next(iter(train_loader))
    print(f"Batch tensor shape: {batch_images.shape}")
    print(f"Batch labels shape: {batch_labels.shape}")

Loading HAM10000 dataset and creating dataloaders...
Total samples: 10015
Training samples: 8012
Testing samples: 2003
Classes: ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
Classes in training set: ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
Classes in testing set: ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
Batch tensor shape: torch.Size([64, 3, 224, 224])
Batch labels shape: torch.Size([64])
Device: cpu
Train samples: 8012
Test samples: 2003
Extracted train feature shape: (8012, 2048)
Extracted train label shape: (8012,)
SVM training finished.
Data loading and model pipeline complete.


In [ ]:
# 1) Extract CNN descriptors and train the SVM model.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
feature_extractor = build_resnet50_feature_extractor(device)

train_features, train_labels = extract_cnn_features(
    train_loader,
    feature_extractor,
    device,
)
test_features, test_labels = extract_cnn_features(
    test_loader,
    feature_extractor,
    device,
)

svm_model = train_svm(train_features, train_labels)

In [ ]:
# 2) Evaluate the trained model on the held-out test split.
test_predictions = svm_model.predict(test_features)

macro_f1 = f1_score(test_labels, test_predictions, average="macro", zero_division=0)
balanced_accuracy = balanced_accuracy_score(test_labels, test_predictions)

label_indices = list(range(len(dataset.class_names)))
per_class_recall = recall_score(
    test_labels,
    test_predictions,
    average=None,
    labels=label_indices,
    zero_division=0,
 )

print("\nTest-set performance (CNN + SVM):")
print(f"Macro F1-score    : {macro_f1:.4f}")
print(f"Balanced accuracy : {balanced_accuracy:.4f}")

print("\nPer-class recall:")
for class_name, class_recall in zip(dataset.class_names, per_class_recall):
    print(f"{class_name:>6s}: {class_recall:.4f}")

cm = confusion_matrix(test_labels, test_predictions, labels=label_indices)
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=dataset.class_names)
disp.plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=False)
ax.set_title("Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# 3) k-fold cross-validation on all extracted CNN descriptors 
all_features = np.concatenate((train_features, test_features), axis=0)
all_labels = np.concatenate((train_labels, test_labels), axis=0)

print("\nK-fold cross-validation (CNN descriptors + SVM):")
for fold in [2, 5, 10]:
    cv = StratifiedKFold(n_splits=fold, shuffle=True, random_state=42)
    cv_scores = cross_val_score(
        build_svm_estimator(),
        all_features,
        all_labels,
        cv=cv,
        scoring="accuracy",
        n_jobs=-1,
    )
    print(f"\nFold = {fold}")
    print(f"Average accuracy: {cv_scores.mean():.4f}")
    print(f"Std deviation   : {cv_scores.std():.4f}")

print("\nData loading, training, and evaluation complete.")

In [ ]:
# 4) Train DenseNet121 with weighted cross-entropy (7 epochs).
train_labels_densenet = train_dataset.dataset.metadata.iloc[train_dataset.indices]["label"].to_numpy()

# Keep this explicit so predict/train always use the same runtime device.
densenet_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

densenet_model, densenet_history = run_densenet_pipeline(
    train_loader=train_loader,
    train_dataset=train_dataset,
    train_labels=train_labels_densenet,
    epochs=7,
)

In [ ]:
# 5) Evaluate DenseNet121 on the held-out test split.
densenet_predictions, densenet_test_labels = predict_densenet(
    model=densenet_model,
    dataloader=test_loader,
    device=densenet_device,
)

densenet_macro_f1 = f1_score(
    densenet_test_labels,
    densenet_predictions,
    average="macro",
    zero_division=0,
)
densenet_balanced_accuracy = balanced_accuracy_score(
    densenet_test_labels,
    densenet_predictions,
)

label_indices = list(range(len(dataset.class_names)))
densenet_per_class_recall = recall_score(
    densenet_test_labels,
    densenet_predictions,
    average=None,
    labels=label_indices,
    zero_division=0,
)

print("\nTest-set performance (DenseNet121):")
print(f"Macro F1-score    : {densenet_macro_f1:.4f}")
print(f"Balanced accuracy : {densenet_balanced_accuracy:.4f}")

print("\nPer-class recall:")
for class_name, class_recall in zip(dataset.class_names, densenet_per_class_recall):
    print(f"{class_name:>6s}: {class_recall:.4f}")

densenet_cm = confusion_matrix(
    densenet_test_labels,
    densenet_predictions,
    labels=label_indices,
)
fig, ax = plt.subplots(figsize=(8, 6))
confusion_display = ConfusionMatrixDisplay(
    confusion_matrix=densenet_cm,
    display_labels=dataset.class_names,
)
confusion_display.plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=False)
ax.set_title("DenseNet121 Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# 6) k-fold cross-validation on DenseNet121 embeddings + SVM.
def extract_densenet_embeddings(model, dataloader, device):
    model.eval()
    all_features = []
    all_labels = []

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            features = model.features(images)
            features = torch.relu(features)
            features = torch.nn.functional.adaptive_avg_pool2d(features, (1, 1))
            features = torch.flatten(features, 1)

            all_features.append(features.cpu().numpy())
            all_labels.append(labels.numpy())

    return np.concatenate(all_features, axis=0), np.concatenate(all_labels, axis=0)


densenet_train_features, densenet_train_labels = extract_densenet_embeddings(
    densenet_model,
    train_loader,
    densenet_device,
)
densenet_test_features, densenet_test_labels_cv = extract_densenet_embeddings(
    densenet_model,
    test_loader,
    densenet_device,
)

all_densenet_features = np.concatenate((densenet_train_features, densenet_test_features), axis=0)
all_densenet_labels = np.concatenate((densenet_train_labels, densenet_test_labels_cv), axis=0)

print("\nK-fold cross-validation (DenseNet121 embeddings + SVM):")
for fold in [2, 5, 10]:
    cv = StratifiedKFold(n_splits=fold, shuffle=True, random_state=42)
    cv_scores = cross_val_score(
        build_svm_estimator(),
        all_densenet_features,
        all_densenet_labels,
        cv=cv,
        scoring="accuracy",
        n_jobs=-1,
    )
    print(f"\nFold = {fold}")
    print(f"Average accuracy: {cv_scores.mean():.4f}")
    print(f"Std deviation   : {cv_scores.std():.4f}")